In [1]:
from math import factorial

from scipy.optimize import minimize_scalar

In [2]:
def y(x:float) -> float:
    return x + 10 / x

In [3]:
x_list = [1.0, 1.5, 2.0, 2.5]
x = 1.4

In [4]:
y_exact = round(y(x), 5)
print("Exact:", y_exact)

Exact: 8.54286


In [5]:
y_list = [round(y(xi), 5) for xi in x_list]
print(y_list)

[11.0, 8.16667, 7.0, 6.5]


In [6]:
def lagrange(x: float, x_list: list[float]) -> float:
    result = y_list.copy()
    for step in range(len(x_list)):
        for i in range(len(x_list)):
            if i == step: continue
            result[step] *= (x - x_list[i]) / (x_list[step] - x_list[i])
    return round(sum(result), 5)

In [7]:
y_interp = lagrange(x, x_list)
print(y_interp)

8.568


In [8]:
absolute_error = abs(y_exact - y_interp)
print(absolute_error)

0.025140000000000384


In [9]:
def diff_y4(x: float) -> float:
    return round(240 / x ** 5, 5)

In [10]:
def multiple(x: float) -> float:
    res = 1
    for i in x_list:
        res *= x - i
    return res

In [11]:
def error_method(x: float) -> float:
    max_diff_y4 = minimize_scalar(lambda x_: diff_y4(x_), bounds=(x_list[0], x_list[-1]), method="bounded").fun
    facorial_4 = factorial(len(x_list))
    return max_diff_y4 / facorial_4 * abs(multiple(x))

In [12]:
print(round(error_method(x), 5))

0.0027


In [13]:
def diff_y1_exact(x: float) -> float:
    return round(1 - 10 / x ** 2, 5)

def diff_y2_exact(x: float) -> float:
    return round(20 / x ** 3, 5)

In [14]:
h = (x_list[-1] - x_list[0]) / 4
print(h)

x_new = [x_list[0] + i * h for i in range(5)]
print(x_new)
y_new = [round(y(xi), 5) for xi in x_new]
print(y_new)

0.375
[1.0, 1.375, 1.75, 2.125, 2.5]
[11.0, 8.64773, 7.46429, 6.83088, 6.5]


In [15]:
def left_diffs(y_list: list[float]) -> list[float]:
    result = []
    for i in range(1, len(y_list)):
        #result.append(round((y_list[i] - y_list[i - 1]) / h, 5))
        result.append((y_list[i] - y_list[i - 1]) / h)
    return result

In [16]:
def right_diffs(y_list: list[float]) -> list[float]:
    result = []
    for i in range(0, len(y_list) - 1):
       # result.append(round((y_list[i + 1] - y_list[i]) / h, 5))
        result.append((y_list[i + 1] - y_list[i]) / h)
    return result

In [17]:
def central_diffs(y_list: list[float]) -> list[float]:
    result = []
    for i in range(1, len(y_list) - 1):
        #result.append(round((y_list[i + 1] - y_list[i - 1]) / (2 * h), 5))
        result.append((y_list[i + 1] - y_list[i - 1]) / (2 * h))
    return result

In [18]:
def second_diffs(y_list: list[float]) -> list[float]:
    result = []
    for i in range(1, len(y_list) - 1):
        result.append(round((y_list[i - 1] - 2 * y_list[i] + y_list[i + 1]) / (h ** 2), 5))
    return result

In [19]:
exact_diffs1 = [diff_y1_exact(xi) for xi in x_new]
exact_diffs2 = [diff_y2_exact(xi) for xi in x_new]

print("Exact first differences", exact_diffs1)
print("Exact second differences", exact_diffs2)

Exact first differences [-9.0, -4.28926, -2.26531, -1.21453, -0.6]
Exact second differences [20.0, 7.69346, 3.73178, 2.08427, 1.28]


In [20]:
left_diffs1 = left_diffs(y_new)
right_diffs1 = right_diffs(y_new)
central_diffs1 = central_diffs(y_new)
second_diffs_not_exact = second_diffs(y_new)

print("Left differences", left_diffs1)
print("Right differences", right_diffs1)
print("Central differences", central_diffs1)
print("Not exact second differences", second_diffs_not_exact)

Left differences [-6.272720000000002, -3.1558399999999978, -1.6890933333333347, -0.8823466666666656]
Right differences [-6.272720000000002, -3.1558399999999978, -1.6890933333333347, -0.8823466666666656]
Central differences [-4.71428, -2.4224666666666663, -1.2857200000000002]
Not exact second differences [8.31168, 3.91132, 2.15132]


In [21]:
print("Left diffs errors", sum([abs(round(left_diffs1[i] - exact_diffs1[i+1], 5)) for i in range(len(exact_diffs1) - 1)]) / len(left_diffs1))
print("Right diffs errors", sum([abs(round(right_diffs1[i] - exact_diffs1[i], 5)) for i in range(len(exact_diffs1) - 1)]) / len(right_diffs1))
print("Central diffs errors", sum([abs(round(central_diffs1[i-1] - exact_diffs1[i], 5)) for i in range(1, len(exact_diffs1)-1)]) / len(central_diffs1))
print("Second diffs errors", sum([abs(round(second_diffs_not_exact[i-1] - exact_diffs2[i], 5)) for i in range(1, len(exact_diffs2) - 1)]) / len(second_diffs_not_exact))

Left diffs errors 0.907725
Right diffs errors 1.192275
Central diffs errors 0.21779
Second diffs errors 0.28826999999999997
